# 01 — EDA & Preprocessing
**Author:** Ahmed Deraz (24046227)  
**Project:** Hourly Urban Air Quality Prediction Using Machine Learning  
**Module:** UFCFAS-15-2 Machine Learning  

**Dataset:** Urban Air Quality & Weather (1,000 daily records, 10 cities, 46 columns)  
**Target:** `Health_Risk_Score` — a composite risk score derived from weather and air quality conditions.

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('../report/figures', exist_ok=True)

TARGET = 'Health_Risk_Score'

df = pd.read_csv('../data/urban_air_quality.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

print(f'Dataset shape : {df.shape}')
print(f'Date range    : {df["datetime"].min().date()} → {df["datetime"].max().date()}')
print(f'Cities        : {sorted(df["City"].unique())}')
print(f'Columns       : {list(df.columns)}')
df.head()

## 2. Basic Statistics

In [ ]:
print('--- Numeric Summary ---')
df.describe()

## 3. Missing Value Analysis

In [ ]:
missing = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_nonzero = missing[missing > 0]

print(f'Columns with missing values: {len(missing_nonzero)}')
print(missing_nonzero.to_string())

plt.figure(figsize=(10, 4))
missing_nonzero.plot(kind='bar', color='coral', edgecolor='black')
plt.axhline(y=40, color='red', linestyle='--', linewidth=1.5, label='40% drop threshold')
plt.title('Missing Values per Column (%)', fontsize=13)
plt.ylabel('Missing (%)')
plt.xlabel('Column')
plt.legend()
plt.tight_layout()
plt.savefig('../report/figures/missing_values.png', dpi=150)
plt.show()
print('Saved → report/figures/missing_values.png')

## 4. Target Variable Distribution (Health_Risk_Score)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df[TARGET], bins=40, color='steelblue', edgecolor='black', alpha=0.85)
axes[0].set_title('Health_Risk_Score Distribution')
axes[0].set_xlabel('Health Risk Score')
axes[0].set_ylabel('Frequency')

axes[1].boxplot(df[TARGET], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Health_Risk_Score Box Plot')
axes[1].set_ylabel('Health Risk Score')

plt.suptitle('Target Variable: Health_Risk_Score', fontsize=13)
plt.tight_layout()
plt.savefig('../report/figures/target_distribution.png', dpi=150)
plt.show()
print('Saved → report/figures/target_distribution.png')
print()
print(df[TARGET].describe())

## 5. Health Risk Score Over Time (by City)

In [ ]:
plt.figure(figsize=(14, 5))
for city in sorted(df['City'].unique()):
    subset = df[df['City'] == city]
    plt.plot(subset['datetime'], subset[TARGET], label=city, linewidth=1, alpha=0.8)

plt.title('Health Risk Score Over Time by City', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Health Risk Score')
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('../report/figures/risk_over_time.png', dpi=150)
plt.show()
print('Saved → report/figures/risk_over_time.png')

## 6. Risk Score by City (Box Plot)

In [ ]:
city_order = df.groupby('City')[TARGET].median().sort_values(ascending=False).index

plt.figure(figsize=(12, 5))
df.boxplot(column=TARGET, by='City', figsize=(12, 5),
           order=city_order, patch_artist=True)
plt.title('Health Risk Score by City')
plt.suptitle('')
plt.xlabel('City')
plt.ylabel('Health Risk Score')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../report/figures/risk_by_city.png', dpi=150)
plt.show()
print('Saved → report/figures/risk_by_city.png')

## 7. Correlation Heatmap (Numeric Features vs Target)

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(
    columns=['datetimeEpoch','sunriseEpoch','sunsetEpoch'], errors='ignore')

corr = numeric_df.corr()

plt.figure(figsize=(16, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            linewidths=0.3, square=False, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('../report/figures/correlation_heatmap.png', dpi=150)
plt.show()
print('Saved → report/figures/correlation_heatmap.png')

print('\nTop 10 features correlated with Health_Risk_Score:')
top_corr = corr[TARGET].drop(TARGET).abs().sort_values(ascending=False).head(10)
print(top_corr.to_string())

## 8. Weather Feature Distributions

In [ ]:
weather_features = ['temp', 'humidity', 'windspeed', 'pressure',
                    'cloudcover', 'visibility', 'uvindex', 'Severity_Score']
weather_features = [f for f in weather_features if f in df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
colors = ['#4e89ae','#ed6663','#43b89c','#f5a623','#7b68ee','#ff69b4','#32cd32','#ffa500']

for i, col in enumerate(weather_features):
    axes[i].hist(df[col].dropna(), bins=40, color=colors[i], edgecolor='black', alpha=0.8)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel('Frequency')

for j in range(len(weather_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Weather Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../report/figures/weather_distributions.png', dpi=150)
plt.show()
print('Saved → report/figures/weather_distributions.png')

## 9. Validate Preprocessing Pipeline

In [ ]:
import sys
sys.path.append('../src')
from preprocessing import load_and_preprocess

X_train, X_test, y_train, y_test, feature_names = load_and_preprocess()

print(f'\n✅ Preprocessing complete!')
print(f'   X_train : {X_train.shape}')
print(f'   X_test  : {X_test.shape}')
print(f'   y_train : {y_train.shape}  range [{y_train.min():.2f}, {y_train.max():.2f}]')
print(f'   y_test  : {y_test.shape}  range [{y_test.min():.2f}, {y_test.max():.2f}]')
print(f'   Features ({len(feature_names)}): {feature_names}')

---
**End of Notebook 01** — EDA complete. Key findings:
- 1,000 daily records across 10 US cities (Sep 2024 – Sep 2025)
- Target `Health_Risk_Score` ranges ~8.5–11.5, normally distributed
- `Severity_Score`, `uvindex`, `temp` are among the strongest correlates with the target
- 2 columns dropped for >40% missingness (`preciptype`, `Condition_Code`)
- 33 features retained after cleaning

**→ Proceed to `02_linear_regression.ipynb`**